# Refactorización y Code Smells

## Introducción
Un *code smell* es un síntoma de diseño fácil de detectar en el código
fuente: no es un error en sí mismo (el programa puede funcionar
perfectamente), pero suele delatar un problema más profundo de diseño.
La **refactorización** es el proceso sistemático de mejorar el diseño
interno del código **sin cambiar su comportamiento externo** — la
herramienta principal para pasar de código sucio a código limpio y,
cuando el smell revela una falla estructural, el paso previo a aplicar
un patrón de diseño.

Este notebook recorre los 13 code smells más comunes cubiertos en el
módulo de clase, organizados en las 5 categorías estándar
(clasificación de refactoring.guru): **Bloaters**, **Object-Orientation
Abusers**, **Change Preventers**, **Dispensables** y **Couplers**. Por
cada uno verás una versión "con el smell", su versión refactorizada y
una explicación de la técnica aplicada.

## Objetivos
- Reconocer visualmente los code smells más frecuentes en código Python.
- Aplicar la técnica de refactorización correspondiente a cada smell.
- Conectar cada smell con la técnica de refactor que lo resuelve, como
  paso de diagnóstico previo a decidir un patrón de diseño.
- Practicar el mismo tipo de análisis que se espera en el diagnóstico
  del "antes" de la Actividad 4 (proyecto integrador).

## Categoría: Bloaters

### 1. Long Method (método largo)

**Definición:** Un método que ha crecido demasiado y asume varias responsabilidades a la vez (validar, calcular, imprimir...).

**Síntoma:** Es psicológicamente más barato añadir 2 líneas a un método existente que crear uno nuevo — el método se vuelve un "Hotel California": entra código, pero nada sale.

**Técnica de refactor:** Extract Method

#### Con el smell

In [ ]:
class Item:
    def __init__(self, nombre, precio, cantidad):
        self.nombre = nombre
        self.precio = precio
        self.cantidad = cantidad

class Pedido:
    def __init__(self, id, items):
        self.id = id
        self.items = items

def generar_reporte(pedido):
    print(f"Reporte de pedido #{pedido.id}")
    total = 0
    for item in pedido.items:
        subtotal = item.precio * item.cantidad
        total += subtotal
        print(f"  {item.nombre}: {subtotal:.2f}")
    if total > 100:
        total *= 0.9  # descuento por volumen
    print(f"TOTAL: {total:.2f}")
    return total

pedido = Pedido(1, [Item("Teclado", 80, 1), Item("Mouse", 30, 2)])
generar_reporte(pedido)

#### Refactorizado

In [ ]:
def _calcular_total(pedido):
    total = sum(item.precio * item.cantidad for item in pedido.items)
    return total * 0.9 if total > 100 else total

def _imprimir_detalle(pedido, total):
    print(f"Reporte de pedido #{pedido.id}")
    for item in pedido.items:
        print(f"  {item.nombre}: {item.precio * item.cantidad:.2f}")
    print(f"TOTAL: {total:.2f}")

def generar_reporte(pedido):
    total = _calcular_total(pedido)
    _imprimir_detalle(pedido, total)
    return total

pedido = Pedido(1, [Item("Teclado", 80, 1), Item("Mouse", 30, 2)])
generar_reporte(pedido)

**Explicación:** `Extract Method` divide `generar_reporte` en dos métodos con una sola responsabilidad cada uno: `_calcular_total` (regla de negocio) e `_imprimir_detalle` (presentación). El resultado impreso es idéntico, pero cada pieza ahora se puede leer, probar y reutilizar por separado.

### 2. Large Class (clase grande)

**Definición:** Una clase que acumula demasiados campos, métodos y responsabilidades — viola SRP.

**Síntoma:** Cada requerimiento nuevo se resuelve agregándole "algo más" a la misma clase, en vez de crear una clase nueva.

**Técnica de refactor:** Extract Class

#### Con el smell

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = salario

    def calcular_impuesto(self):
        return self.salario * 0.19

    def calcular_bono(self, meses):
        return self.salario * 0.1 * meses

    def imprimir_recibo(self):
        imp = self.calcular_impuesto()
        neto = self.salario - imp
        print(f"{self.nombre}: neto {neto:.2f}")

empleado = Empleado("Ana", 3_000_000)
empleado.imprimir_recibo()

#### Refactorizado

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = salario

class CalculadoraNomina:
    def impuesto(self, empleado):
        return empleado.salario * 0.19

    def bono(self, empleado, meses):
        return empleado.salario * 0.1 * meses

class ReciboImpresora:
    def imprimir(self, empleado, calculadora):
        imp = calculadora.impuesto(empleado)
        neto = empleado.salario - imp
        print(f"{empleado.nombre}: neto {neto:.2f}")

empleado = Empleado("Ana", 3_000_000)
calc = CalculadoraNomina()
ReciboImpresora().imprimir(empleado, calc)

**Explicación:** `Extract Class` separa datos (`Empleado`), reglas de nómina (`CalculadoraNomina`) e impresión (`ReciboImpresora`) en tres clases con una sola razón para cambiar cada una.

### 3. Primitive Obsession (obsesión primitiva)

**Definición:** Usar tipos primitivos (str, int, listas) para representar conceptos de dominio (teléfono, dinero, dirección) en vez de una clase dedicada.

**Síntoma:** "Es solo un campo simple para guardar un dato" — hasta que la validación y el formateo de ese dato se repiten en cada función que lo usa.

**Técnica de refactor:** Replace Data Value with Object

#### Con el smell

In [ ]:
class Cliente:
    def __init__(self, nombre, telefono):
        self.nombre = nombre
        self.telefono = telefono  # str sin validar

def enviar_sms(telefono):
    limpio = telefono.replace("-", "")
    if len(limpio) != 10:
        raise ValueError("Teléfono inválido")
    print(f"SMS enviado a {limpio}")

cliente = Cliente("Ana", "300-555-1234")
enviar_sms(cliente.telefono)

#### Refactorizado

In [ ]:
class Telefono:
    def __init__(self, numero):
        limpio = numero.replace("-", "")
        if len(limpio) != 10:
            raise ValueError("Teléfono inválido")
        self._numero = limpio

    def enviar_sms(self):
        print(f"SMS enviado a {self._numero}")

class Cliente:
    def __init__(self, nombre, telefono):
        self.nombre = nombre
        self.telefono = Telefono(telefono)

cliente = Cliente("Ana", "300-555-1234")
cliente.telefono.enviar_sms()

**Explicación:** `Replace Data Value with Object` crea `Telefono` como clase dedicada: la validación vive en un solo lugar, y cualquier otra parte del sistema que necesite enviar SMS reutiliza el mismo método en vez de reimplementar la limpieza del número.

## Categoría: Object-Orientation Abusers

### 4. Switch Statements (condicionales por tipo)

**Definición:** Condicionales (if/elif o switch) que ramifican según el "tipo" de un objeto, repetidos en varios lugares del código.

**Síntoma:** Cada figura/tipo nuevo obliga a editar el mismo if/elif en todos los lugares donde aparezca esa lógica.

**Técnica de refactor:** Replace Conditional with Polymorphism

#### Con el smell

In [ ]:
def calcular_area(figura):
    if figura["tipo"] == "circulo":
        return 3.1416 * figura["radio"] ** 2
    elif figura["tipo"] == "cuadrado":
        return figura["lado"] ** 2
    elif figura["tipo"] == "rectangulo":
        return figura["base"] * figura["altura"]
    raise ValueError("Figura desconocida")

figuras = [
    {"tipo": "circulo", "radio": 2},
    {"tipo": "cuadrado", "lado": 3},
]
for f in figuras:
    print(f'{f["tipo"]}: {calcular_area(f):.2f}')

#### Refactorizado

In [ ]:
class Figura:
    def area(self):
        raise NotImplementedError

class Circulo(Figura):
    def __init__(self, radio):
        self.radio = radio

    def area(self):
        return 3.1416 * self.radio ** 2

class Cuadrado(Figura):
    def __init__(self, lado):
        self.lado = lado

    def area(self):
        return self.lado ** 2

figuras = [Circulo(2), Cuadrado(3)]
for f in figuras:
    print(f"{type(f).__name__}: {f.area():.2f}")

**Explicación:** `Replace Conditional with Polymorphism` mueve cada rama del condicional a un método `area()` propio de cada subclase. Agregar una figura nueva ya no exige tocar ningún if/elif existente: solo se crea una subclase más.

### 5. Refused Bequest (legado rechazado)

**Definición:** Una subclase hereda métodos de una superclase pero no los usa — o los sobreescribe solo para lanzar una excepción.

**Síntoma:** La jerarquía de herencia no refleja una relación real "es-un": obliga a la subclase a cargar con comportamiento que no le corresponde.

**Técnica de refactor:** Replace Inheritance with Delegation / reordenar la jerarquía

#### Con el smell

In [ ]:
class Ave:
    def volar(self):
        print("Volando alto")

class Pinguino(Ave):
    def volar(self):
        # el pingüino rechaza el legado de Ave
        raise NotImplementedError("Los pingüinos no vuelan")

aves = [Ave(), Pinguino()]
for ave in aves:
    try:
        ave.volar()
    except NotImplementedError as e:
        print(f"Error: {e}")

#### Refactorizado

In [ ]:
class Ave:
    def nadar(self):
        print("Nadando")

class AveVoladora(Ave):
    def volar(self):
        print("Volando alto")

class Pinguino(Ave):
    pass  # no hereda volar(): simplemente no aplica

aves = [AveVoladora(), Pinguino()]
for ave in aves:
    ave.nadar()
    if isinstance(ave, AveVoladora):
        ave.volar()

**Explicación:** Se reordena la jerarquía: `Ave` solo contiene lo común a todas las aves (`nadar`), y `AveVoladora` agrega `volar()` solo donde aplica. `Pinguino` ya no necesita rechazar nada porque nunca heredó un método que no puede cumplir.

## Categoría: Change Preventers

### 6. Shotgun Surgery (cirugía de escopeta)

**Definición:** Un solo cambio de regla de negocio obliga a modificar pequeñas partes en muchas clases distintas del sistema.

**Síntoma:** La misma constante o regla de negocio está copiada en múltiples clases no relacionadas.

**Técnica de refactor:** Move Method / Move Field (centralizar la regla)

#### Con el smell

In [ ]:
class Factura:
    def total_con_iva(self, subtotal):
        return subtotal * 1.19  # 19% quemado

class Reporte:
    def calcular_iva(self, monto):
        return monto * 0.19  # duplicado

class Carrito:
    def impuesto(self, monto):
        return monto * 0.19  # duplicado otra vez

print(Factura().total_con_iva(100))
print(Reporte().calcular_iva(100))
print(Carrito().impuesto(100))

#### Refactorizado

In [ ]:
class ConfiguracionImpuestos:
    TASA_IVA = 0.19

    @classmethod
    def iva(cls, monto):
        return monto * cls.TASA_IVA

class Factura:
    def total_con_iva(self, subtotal):
        return subtotal + ConfiguracionImpuestos.iva(subtotal)

class Reporte:
    def calcular_iva(self, monto):
        return ConfiguracionImpuestos.iva(monto)

class Carrito:
    def impuesto(self, monto):
        return ConfiguracionImpuestos.iva(monto)

print(Factura().total_con_iva(100))
print(Reporte().calcular_iva(100))
print(Carrito().impuesto(100))

**Explicación:** `ConfiguracionImpuestos` concentra la regla de negocio (la tasa de IVA). Cambiar el IVA de 19% a 20% ahora significa editar una sola línea, en un solo lugar, en vez de perseguir cada copia por el código.

### 7. Divergent Change (cambio divergente)

**Definición:** Una misma clase cambia de formas totalmente distintas según cuál sea la razón del cambio.

**Síntoma:** Modificar una regla de nómina y modificar el motor de persistencia terminan tocando la misma clase — dos razones de cambio mezcladas.

**Técnica de refactor:** Extract Class

#### Con el smell

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = salario

    def calcular_pago_mensual(self):
        return self.salario / 12

    def guardar_en_base_datos(self):
        # cambia si cambia el motor de BD, no la nómina
        print(f"INSERT INTO empleados VALUES ('{self.nombre}')")

empleado = Empleado("Ana", 3_600_000)
print(empleado.calcular_pago_mensual())
empleado.guardar_en_base_datos()

#### Refactorizado

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = salario

class CalculadoraSalario:
    def pago_mensual(self, empleado):
        return empleado.salario / 12

class EmpleadoRepositorio:
    def guardar(self, empleado):
        print(f"INSERT INTO empleados VALUES ('{empleado.nombre}')")

empleado = Empleado("Ana", 3_600_000)
print(CalculadoraSalario().pago_mensual(empleado))
EmpleadoRepositorio().guardar(empleado)

**Explicación:** `Extract Class` separa las dos razones de cambio: `CalculadoraSalario` cambia solo si cambian las reglas de nómina, `EmpleadoRepositorio` cambia solo si cambia el motor de persistencia. `Empleado` ya no cambia por ninguna de las dos razones.

## Categoría: Dispensables

### 8. Duplicate Code (código duplicado) — PRIORIDAD #1

**Definición:** La misma estructura de código (idéntica o casi idéntica) aparece repetida en más de un lugar del proyecto.

**Síntoma:** "El código limpio no contiene duplicación": cada copia es un lugar más donde hay que recordar aplicar el mismo cambio.

**Técnica de refactor:** Extract Method

#### Con el smell

In [ ]:
def precio_regular(base):
    descuento = base * 0.05
    if base > 100:
        descuento += base * 0.02
    return base - descuento

def precio_vip(base):
    descuento = base * 0.05
    if base > 100:
        descuento += base * 0.02
    descuento += base * 0.03  # descuento extra VIP
    return base - descuento

print(precio_regular(150))
print(precio_vip(150))

#### Refactorizado

In [ ]:
def _descuento_base(base):
    descuento = base * 0.05
    if base > 100:
        descuento += base * 0.02
    return descuento

def precio_regular(base):
    return base - _descuento_base(base)

def precio_vip(base):
    descuento = _descuento_base(base) + base * 0.03
    return base - descuento

print(precio_regular(150))
print(precio_vip(150))

**Explicación:** `Extract Method` saca la lógica de descuento compartida a `_descuento_base`. Si mañana cambia la regla del descuento base, se edita una sola función y ambos precios quedan actualizados automáticamente.

### 9. Dead Code (código muerto)

**Definición:** Variables, métodos o clases que ya no se usan en ninguna parte del sistema.

**Síntoma:** Sobrevive a una migración o refactor anterior porque nadie se atrevió a borrarlo "por si acaso".

**Técnica de refactor:** Eliminar directamente (el control de versiones es el respaldo)

#### Con el smell

In [ ]:
def calcular_envio(peso):
    return peso * 1500

def calcular_envio_legacy(peso, zona):
    # tarifa antigua, ya no se usa desde que se
    # migró a calcular_envio(); nadie la llama
    if zona == "urbana":
        return peso * 1200
    return peso * 1800

total = calcular_envio(5)
print(total)

#### Refactorizado

In [ ]:
def calcular_envio(peso):
    return peso * 1500

# calcular_envio_legacy() se eliminó: no había
# ninguna llamada en el código ni en las pruebas.
# El historial de git conserva la versión anterior
# si en algún momento hace falta consultarla.

total = calcular_envio(5)
print(total)

**Explicación:** El código muerto se elimina directamente, sin comentarlo ni "guardarlo por si acaso": el sistema de control de versiones ya cumple esa función, y cada línea de código vivo que se mantiene es una línea que hay que seguir leyendo y entendiendo.

### 10. Speculative Generality (generalidad especulativa)

**Definición:** Abstracciones (clases abstractas, hooks, parámetros) creadas anticipadamente "por si se necesitan en el futuro", que nunca llegan a usarse.

**Síntoma:** Solo existe una subclase real, pero la jerarquía está preparada para muchas más que jamás aparecieron.

**Técnica de refactor:** Collapse Hierarchy

#### Con el smell

In [ ]:
class ProcesadorAbstracto:
    def preprocesar(self, datos):
        return datos  # hook nunca usado

    def procesar(self, datos):
        raise NotImplementedError

    def posprocesar(self, resultado):
        return resultado  # hook nunca usado

class ProcesadorCSV(ProcesadorAbstracto):
    def procesar(self, datos):
        return datos.strip().split(",")

p = ProcesadorCSV()
print(p.procesar("a, b, c"))

#### Refactorizado

In [ ]:
class ProcesadorCSV:
    def procesar(self, datos):
        return datos.strip().split(",")

# Se eliminó la superclase especulativa: nunca
# existió un segundo procesador ni se necesitaron
# los hooks pre/posprocesar().

p = ProcesadorCSV()
print(p.procesar("a, b, c"))

**Explicación:** `Collapse Hierarchy` fusiona la única subclase real con su superclase especulativa. Si en el futuro aparece un segundo procesador real, se extrae la abstracción común en ese momento — no antes.

## Categoría: Couplers

### 11. Feature Envy (envidia de características)

**Definición:** Un método de una clase está más interesado en los datos de otra clase que en los de la suya propia.

**Síntoma:** El método recorre y opera directamente sobre los atributos internos de un objeto ajeno.

**Técnica de refactor:** Move Method

#### Con el smell

In [ ]:
class Carrito:
    def __init__(self, items):
        self.items = items  # lista de (precio, cantidad)

class Factura:
    def calcular_total(self, carrito):
        total = 0
        for precio, cantidad in carrito.items:
            total += precio * cantidad
        return total  # envidia los datos de Carrito

carrito = Carrito([(80, 1), (30, 2)])
print(Factura().calcular_total(carrito))

#### Refactorizado

In [ ]:
class Carrito:
    def __init__(self, items):
        self.items = items

    def calcular_total(self):
        return sum(precio * cantidad for precio, cantidad in self.items)

class Factura:
    def calcular_total(self, carrito):
        return carrito.calcular_total()

carrito = Carrito([(80, 1), (30, 2)])
print(Factura().calcular_total(carrito))

**Explicación:** `Move Method` traslada el cálculo a `Carrito`, la clase dueña de los datos. `Factura` ahora solo pide el resultado en vez de recorrer datos ajenos.

### 12. Message Chains (cadenas de mensajes)

**Definición:** Una secuencia de llamadas encadenadas del tipo `a.getB().getC().getD()` que expone toda la estructura interna de una jerarquía de objetos.

**Síntoma:** El código cliente necesita conocer Cliente → Dirección → Ciudad solo para leer un nombre.

**Técnica de refactor:** Hide Delegate

#### Con el smell

In [ ]:
class Ciudad:
    def __init__(self, nombre):
        self.nombre = nombre

class Direccion:
    def __init__(self, ciudad):
        self.ciudad = ciudad

class Cliente:
    def __init__(self, direccion):
        self.direccion = direccion

cliente = Cliente(Direccion(Ciudad("Bogotá")))
nombre_ciudad = cliente.direccion.ciudad.nombre  # cadena de mensajes
print(nombre_ciudad)

#### Refactorizado

In [ ]:
class Ciudad:
    def __init__(self, nombre):
        self.nombre = nombre

class Direccion:
    def __init__(self, ciudad):
        self.ciudad = ciudad

class Cliente:
    def __init__(self, direccion):
        self.direccion = direccion

    def ciudad(self):  # oculta la cadena interna
        return self.direccion.ciudad.nombre

cliente = Cliente(Direccion(Ciudad("Bogotá")))
print(cliente.ciudad())

**Explicación:** `Hide Delegate` agrega un método `ciudad()` en `Cliente` que oculta la cadena. El código cliente ya no necesita saber que existen `Direccion` y `Ciudad` como pasos intermedios.

### 13. Inappropriate Intimacy (intimidad inapropiada)

**Definición:** Dos clases dependen de forma tan estrecha que una manipula directamente los detalles internos ("privados") de la otra.

**Síntoma:** Una clase modifica atributos con guion bajo (`_saldo`) de otra clase en lugar de pedirle que lo haga ella misma.

**Técnica de refactor:** Move Method / encapsular el estado

#### Con el smell

In [ ]:
class CuentaBancaria:
    def __init__(self, saldo):
        self._saldo = saldo  # "privado" por convención

class Banco:
    def transferir(self, origen, destino, monto):
        # accede directo al interno de la cuenta
        origen._saldo -= monto
        destino._saldo += monto

a = CuentaBancaria(1000)
b = CuentaBancaria(0)
Banco().transferir(a, b, 300)
print(a._saldo, b._saldo)

#### Refactorizado

In [ ]:
class CuentaBancaria:
    def __init__(self, saldo):
        self._saldo = saldo

    def retirar(self, monto):
        self._saldo -= monto

    def depositar(self, monto):
        self._saldo += monto

    def saldo(self):
        return self._saldo

class Banco:
    def transferir(self, origen, destino, monto):
        origen.retirar(monto)
        destino.depositar(monto)

a = CuentaBancaria(1000)
b = CuentaBancaria(0)
Banco().transferir(a, b, 300)
print(a.saldo(), b.saldo())

**Explicación:** `CuentaBancaria` ahora protege su propio estado a través de `retirar()` y `depositar()`. `Banco` ya no conoce ni toca el atributo interno `_saldo` de otra clase: la intimidad inapropiada desaparece.

## Ejercicios y autoevaluación

1. **Identifica el smell**: toma una clase de tu propio proyecto y
   respóndete: ¿tiene más de 5-6 métodos? ¿mezcla más de una
   responsabilidad? Si es así, ¿qué categoría de smell aplica —
   Bloater (Large Class) o Change Preventer (Divergent Change)?
2. **Refactoriza tú mismo**: el siguiente código tiene un `Long
   Parameter List` (Bloater) que no se cubrió en detalle arriba.
   Refactorízalo con `Introduce Parameter Object`, creando una clase
   `Rectangulo` que agrupe `ancho` y `alto`.

```python
def calcular_area_rectangulo(ancho, alto):
    return ancho * alto

def calcular_perimetro_rectangulo(ancho, alto):
    return 2 * (ancho + alto)
```

3. **Detecta un smell real**: busca en tu propio proyecto un `if/elif`
   que ramifique según un campo `tipo` o similar. ¿Podrías resolverlo
   con `Replace Conditional with Polymorphism`? Descríbelo en un
   párrafo, sin necesidad de implementarlo todavía.
4. **Autoevaluación rápida**: sin mirar el catálogo, intenta nombrar
   de memoria 5 code smells y su categoría. Verifica tus respuestas
   contra la tabla resumen del deck de la sesión.
5. **Conexión con tu proyecto (Actividad 4)**: elige un archivo de tu
   proyecto final y lista, en una tabla, cada code smell que
   encuentres, su categoría y la técnica de refactor que aplicarías.
   Ese ejercicio es exactamente el diagnóstico del "antes" que pide la
   rúbrica (`docs/rubricas/proyecto.md`).

## Referencias
- Fowler, M. (2018). *Refactoring: Improving the Design of Existing
  Code* (2nd ed.). Addison-Wesley.
- Refactoring.Guru — [Catálogo de Code Smells](https://refactoring.guru/es/refactoring/smells)
- Refactoring.Guru — [Catálogo de técnicas de refactorización](https://refactoring.guru/es/refactoring/techniques)
- Cunningham, W. (1992). *The WyCash Portfolio Management System*
  (origen de la metáfora de la deuda técnica).
- Material del curso: notebook NotebookLM "Diseño de Patrones
  (Especialización)", sección de refactorización y code smells.